In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
os.getenv('OPENAI_API_KEY')[-10:]

## ReAct와 LangGraph의 연결 방식

ReAct는 Reason과 Act를 번갈아 수행하는 패턴이다. 질문을 받으면 먼저 현재 답변에 필요한 정보를 판단하고, 부족하면 도구를 호출한다. 그 뒤 도구 결과를 반영해 다시 생각한다.

아래 코드는 이 흐름을 `think -> act -> answer`의 순서로 분해한다. 상태에는 질문, 생각 메모, 도구 결과, 최종 답변만 둬서 흐름이 어떻게 이어지는지 쉽게 보이도록 만든다.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START, END

class ReActState(TypedDict):
    question:str
    thought:str
    tool_result:str
    answer:str

def think(state:ReActState):
    question = state['question']
    if '문서' in question or '근거' in question:
        thought = '검색도구가 필요합니다..'
    else:
        thought = '바로 답할수 있습니다.'
    return {'thought':thought}

def act(state:ReActState):
    question = state['question']
    if '문서' in question or '근거' in question:
        tool_result = '벡터DB에서 관련 문단 2개를 찾았습니다.'
    else:
        tool_result = '외부도구가 필요하지 않다.'
    return {'tool_result':tool_result}
def answer(state:ReActState):
    answer = f"질문:{state['question']}\n생각:{state['thought']}\n도구:{state['tool_result']}"
    return {'answer':answer}

workflow = StateGraph(ReActState)
workflow.add_node('think',think)
workflow.add_node('act',act)
workflow.add_node('answer',answer)
workflow.add_edge(START,'think')
workflow.add_edge('think','act')
workflow.add_edge('act','answer')
workflow.add_edge('answer',END)
app = workflow.compile()
kwargs = {
    'question':'문서 근거가 피요한 더미 질문입니다.',
    'thought':'',
    'tool_result':'',
    'answer':''
}
result = app.invoke(kwargs)
result

InvalidUpdateError: Expected dict, got 질문:문서 근거가 피요한 더미 질문입니다.
생각:검색도구가 필요합니다..
도구:벡터DB에서 관련 문단 2개를 찾았습니다.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE